# GNN Layer-by-Layer Tutorial: Polyketide Classifier

This notebook steps through **every layer** of the trained GAT-based polyketide classifier,
explaining the mathematics and showing tensor shapes at each stage. By the end, you will
understand exactly how a SMILES string becomes a PKS/non-PKS probability.

**Architecture overview:**
```
SMILES string
    │
    ▼  RDKit parsing + featurization
Node features  [N, 40]     (one-hot atom properties)
Edge index     [2, E]      (bond connectivity + self-loops)
Edge features  [E, 5]      (one-hot bond types + self-loop marker)
    │
    ▼  Linear projection
Node embeddings [N, 256]
    │
    ▼  GAT layer 0  (4 heads × 256 dim, averaged → 256, residual + LayerNorm + ELU)
    ▼  GAT layer 1  (4 heads × 256 dim, averaged → 256, residual + LayerNorm + ELU)
    ▼  GAT layer 2  (4 heads × 256 dim, averaged → 256, residual + LayerNorm + ELU)
Node embeddings [N, 256]
    │
    ▼  Global mean pooling
Graph embedding [1, 256]
    │
    ▼  Classification head: Linear(256→128) → ReLU → Linear(128→1)
Logit [1, 1]  →  sigmoid  →  P(PKS)
```

**Total parameters:** ~840K

## Roadmap

0. **Title & Motivation** — what this notebook covers
1. **Setup & Checkpoint Loading** — imports, load trained model
2. **SMILES → Molecular Graph** — atom/bond featurization, graph construction
3. **Input Projection** — linear mapping from 40-dim to 256-dim
4. **Graph Attention Layer Deep Dive** — attention mechanism math, manual computation
5. **Full GAT Block** — residual connection, LayerNorm, ELU activation
6. **GAT Layers 1 & 2** — running all 3 layers, embedding evolution
7. **Mean Pooling** — nodes → graph-level embedding
8. **Classification Head** — MLP + sigmoid
9. **End-to-End Verification** — manual vs model forward pass
10. **Understanding the Loss** — BCEWithLogitsLoss with pos_weight
11. **Batched Processing** — how PyG-style batching works
12. **Summary** — recap table of all layers

---
## Section 1: Setup & Checkpoint Loading

In [ ]:
import sys
from pathlib import Path
from typing import Dict, Iterable, List, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from rdkit import Chem, RDLogger
from rdkit.Chem import rdchem, Draw
RDLogger.DisableLog("rdApp.*")

# ---------- Model architecture (copied from scripts/08_train_gnn_classifier.py) ----------

def edge_softmax(dst: torch.Tensor, scores: torch.Tensor, num_nodes: int) -> torch.Tensor:
    """Softmax over edges grouped by destination node."""
    heads = scores.size(1)
    out = []
    for h in range(heads):
        s = scores[:, h]
        max_vals = torch.full((num_nodes,), -float("inf"), device=s.device)
        max_vals.scatter_reduce_(0, dst, s, reduce="amax")
        s = s - max_vals[dst]
        exp_s = torch.exp(s)
        denom = torch.zeros(num_nodes, device=s.device).scatter_add_(0, dst, exp_s)
        out.append(exp_s / (denom[dst] + 1e-16))
    return torch.stack(out, dim=1)


class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads, edge_dim, dropout=0.0):
        super().__init__()
        self.heads = heads
        self.out_dim = out_dim
        self.lin = nn.Linear(in_dim, out_dim * heads, bias=False)
        self.att_src = nn.Parameter(torch.Tensor(heads, out_dim))
        self.att_dst = nn.Parameter(torch.Tensor(heads, out_dim))
        self.bias = nn.Parameter(torch.Tensor(out_dim))
        self.edge_proj = nn.Linear(edge_dim, heads, bias=False)
        self.dropout = dropout
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        nn.init.zeros_(self.bias)
        nn.init.xavier_uniform_(self.edge_proj.weight)

    def forward(self, x, edge_index, edge_attr):
        h = self.lin(x)                                         # [N, heads*out_dim]
        num_nodes = h.size(0)
        h = h.view(num_nodes, self.heads, self.out_dim)         # [N, heads, out_dim]
        att_src = (h * self.att_src).sum(dim=-1)                # [N, heads]
        att_dst = (h * self.att_dst).sum(dim=-1)                # [N, heads]
        src, dst = edge_index
        edge_logits = att_src[src] + att_dst[dst] + self.edge_proj(edge_attr)  # [E, heads]
        edge_logits = F.leaky_relu(edge_logits, 0.2)
        edge_alpha = edge_softmax(dst, edge_logits, num_nodes)  # [E, heads]
        edge_alpha = F.dropout(edge_alpha, p=self.dropout, training=self.training)
        out = h[src] * edge_alpha.unsqueeze(-1)                 # [E, heads, out_dim]
        agg = torch.zeros(num_nodes, self.heads, self.out_dim, device=x.device)
        agg.index_add_(0, dst, out)
        return agg.mean(dim=1) + self.bias                     # [N, out_dim]


class SupervisedGNNClassifier(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=256, heads=4, num_layers=3, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.input_proj = nn.Linear(node_dim, hidden_dim)
        self.gat_layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        for _ in range(num_layers):
            self.gat_layers.append(GraphAttentionLayer(hidden_dim, hidden_dim, heads, edge_dim, dropout))
            self.layer_norms.append(nn.LayerNorm(hidden_dim))
        self.dropout = dropout
        self.cls_head = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
        )

    def forward(self, node_feat, edge_index, batch, edge_attr):
        x = self.input_proj(node_feat)
        for gat, norm in zip(self.gat_layers, self.layer_norms):
            residual = x
            x = gat(x, edge_index, edge_attr)
            x = norm(x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + residual
        num_graphs = int(batch.max().item()) + 1 if batch.numel() > 0 else 0
        graph_embed = torch.zeros(num_graphs, self.hidden_dim, device=x.device)
        counts = torch.zeros(num_graphs, device=x.device)
        graph_embed.index_add_(0, batch, x)
        counts.index_add_(0, batch, torch.ones_like(batch, dtype=x.dtype))
        graph_embed = graph_embed / counts.clamp_min(1.0).unsqueeze(-1)
        logit = self.cls_head(graph_embed)
        return logit, graph_embed


# ---------- Checkpoint loader (from scripts/11_run_inference.py) ----------

def load_model_from_checkpoint(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model_state = checkpoint["model_state_dict"]
    if "args" in checkpoint:
        sa = checkpoint["args"]
        node_dim = model_state["input_proj.weight"].shape[1]
        edge_dim = model_state["gat_layers.0.edge_proj.weight"].shape[1]
        model = SupervisedGNNClassifier(
            node_dim=node_dim, edge_dim=edge_dim,
            hidden_dim=sa["hidden_dim"], heads=sa["num_heads"],
            num_layers=sa["num_layers"], dropout=sa["dropout"],
        )
    else:
        hidden_dim = model_state["input_proj.weight"].shape[0]
        node_dim = model_state["input_proj.weight"].shape[1]
        edge_dim = model_state["gat_layers.0.edge_proj.weight"].shape[1]
        num_layers = sum(1 for k in model_state if k.startswith("gat_layers.") and k.endswith(".lin.weight"))
        heads = model_state["gat_layers.0.att_src"].shape[0]
        model = SupervisedGNNClassifier(
            node_dim=node_dim, edge_dim=edge_dim,
            hidden_dim=hidden_dim, heads=heads, num_layers=num_layers,
        )
    model.load_state_dict(model_state)
    model.to(device)
    model.eval()
    epoch = checkpoint.get("epoch", "?")
    val_loss = checkpoint.get("val_loss", float("nan"))
    print(f"Loaded checkpoint from epoch {epoch} (val loss: {val_loss:.4f})")
    return model


# ---------- Load model ----------

device = torch.device("cpu")  # CPU is fine for a single-molecule tutorial
CHECKPOINT = "../models/supervised_gnn/best_model.pt"
model = load_model_from_checkpoint(CHECKPOINT, device)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"\nModel architecture:\n{model}")

The checkpoint stores two things: a `model_state_dict` (all learned weight tensors) and `args`
(the hyperparameters used during training — hidden dim, number of heads, etc.). We reconstruct
the model architecture from `args`, then load the weights.

---
## Section 2: SMILES → Molecular Graph

A molecule is naturally a graph: **atoms are nodes** and **bonds are edges**. We convert a
SMILES string to three tensors:

| Tensor | Shape | Description |
|--------|-------|-------------|
| `node_feat` | `[N, 40]` | One-hot encoded atom properties |
| `edge_index` | `[2, E]` | Source→destination pairs (bidirectional + self-loops) |
| `edge_attr` | `[E, 5]` | One-hot bond type (SINGLE/DOUBLE/TRIPLE/AROMATIC/self-loop) |

Each atom is described by a **40-dimensional feature vector** built from these one-hot groups:

| Feature | Dims | Values |
|---------|------|--------|
| Atomic number | 13 | H, B, C, N, O, F, Si, P, S, Cl, Br, I + unknown |
| Degree | 7 | 0–5 + unknown |
| Formal charge | 6 | -2 to +2 + unknown |
| Num hydrogens | 6 | 0–4 + unknown |
| Hybridization | 6 | SP, SP2, SP3, SP3D, SP3D2 + unknown |
| Is aromatic | 1 | binary |
| Is in ring | 1 | binary |
| **Total** | **40** | |

In [ ]:
# ---------- Featurization functions (from scripts/08_train_gnn_classifier.py) ----------

ATOM_TYPES = [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]
DEGREES = [0, 1, 2, 3, 4, 5]
FORMAL_CHARGES = [-2, -1, 0, 1, 2]
NUM_HS = [0, 1, 2, 3, 4]
HYBRIDIZATIONS = [
    rdchem.HybridizationType.SP,
    rdchem.HybridizationType.SP2,
    rdchem.HybridizationType.SP3,
    rdchem.HybridizationType.SP3D,
    rdchem.HybridizationType.SP3D2,
]
BOND_TYPES = [
    rdchem.BondType.SINGLE,
    rdchem.BondType.DOUBLE,
    rdchem.BondType.TRIPLE,
    rdchem.BondType.AROMATIC,
]

def _build_mapping(values):
    return {value: idx for idx, value in enumerate(values)}

ATOM_MAP = _build_mapping(ATOM_TYPES)
DEGREE_MAP = _build_mapping(DEGREES)
CHARGE_MAP = _build_mapping(FORMAL_CHARGES)
NUM_H_MAP = _build_mapping(NUM_HS)
HYB_MAP = {hyb: idx for idx, hyb in enumerate(HYBRIDIZATIONS)}
BOND_MAP = {bond: idx for idx, bond in enumerate(BOND_TYPES)}
EDGE_FEAT_DIM = len(BOND_TYPES) + 1  # 5

def _one_hot(value, mapping):
    size = len(mapping) + 1
    vec = np.zeros(size, dtype=np.float32)
    vec[mapping.get(value, len(mapping))] = 1.0
    return vec

def atom_to_feature(atom):
    feats = [
        _one_hot(atom.GetAtomicNum(), ATOM_MAP),                        # 13 dim
        _one_hot(atom.GetTotalDegree(), DEGREE_MAP),                    # 7 dim
        _one_hot(atom.GetFormalCharge(), CHARGE_MAP),                   # 6 dim
        _one_hot(atom.GetTotalNumHs(includeNeighbors=True), NUM_H_MAP), # 6 dim
        _one_hot(atom.GetHybridization(), HYB_MAP),                    # 6 dim
        np.array([atom.GetIsAromatic()], dtype=np.float32),            # 1 dim
        np.array([atom.IsInRing()], dtype=np.float32),                 # 1 dim
    ]
    return np.concatenate(feats, axis=0)  # shape: [40]

def bond_to_feature(bond):
    vec = np.zeros(EDGE_FEAT_DIM, dtype=np.float32)
    if bond is None:
        vec[-1] = 1.0  # self-loop
    else:
        vec[BOND_MAP.get(bond.GetBondType(), EDGE_FEAT_DIM - 1)] = 1.0
    return vec  # shape: [5]

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    n = mol.GetNumAtoms()
    node_feat = np.vstack([atom_to_feature(a) for a in mol.GetAtoms()]).astype(np.float32)
    edges, edge_feat = [], []
    for bond in mol.GetBonds():
        u, v = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat = bond_to_feature(bond)
        edges += [(u, v), (v, u)]
        edge_feat += [feat, feat]
    loop = bond_to_feature(None)
    for i in range(n):
        edges.append((i, i))
        edge_feat.append(loop)
    edge_index = np.array(edges, dtype=np.int64).T  # shape: [2, E]
    edge_attr = np.vstack(edge_feat).astype(np.float32)  # shape: [E, 5]
    return node_feat, edge_index, edge_attr


# ---------- Featurize aspirin ----------
ASPIRIN_SMILES = "CC(=O)OC1=CC=CC=C1C(=O)O"  # 13 heavy atoms
mol = Chem.MolFromSmiles(ASPIRIN_SMILES)
print(f"Molecule: aspirin")
print(f"SMILES:   {ASPIRIN_SMILES}")
print(f"Atoms:    {mol.GetNumAtoms()}")
print(f"Bonds:    {mol.GetNumBonds()}")
print()

# Show per-atom features
node_feat, edge_index, edge_attr = smiles_to_graph(ASPIRIN_SMILES)
N = node_feat.shape[0]
E = edge_index.shape[1]
num_bonds = mol.GetNumBonds()

print(f"node_feat  shape: [{N}, {node_feat.shape[1]}]")
print(f"edge_index shape: [2, {E}]  (= 2×{num_bonds} bidirectional bond edges + {N} self-loops)")
print(f"edge_attr  shape: [{E}, {edge_attr.shape[1]}]")
print()

# Show atom symbols and their feature dimensions
atom_symbols = [mol.GetAtomWithIdx(i).GetSymbol() for i in range(N)]
print("Atom index → symbol:")
for i, sym in enumerate(atom_symbols):
    print(f"  {i:2d} → {sym}", end="")
    if (i + 1) % 5 == 0:
        print()
print()

In [ ]:
# Visualization 1: Atom feature matrix heatmap
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(node_feat, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_yticks(range(N))
ax.set_yticklabels([f"{i}: {sym}" for i, sym in enumerate(atom_symbols)])
ax.set_xlabel("Feature dimension")
ax.set_ylabel("Atom")
ax.set_title(f"Atom feature matrix  [{N}, 40]\n"
             "(atomic_num [0:13] | degree [13:20] | charge [20:26] | "
             "num_Hs [26:32] | hybrid [32:38] | arom [38] | ring [39])")

# Add vertical lines to separate feature groups
for boundary in [13, 20, 26, 32, 38, 39]:
    ax.axvline(x=boundary - 0.5, color="black", linewidth=0.8, linestyle="--", alpha=0.5)

plt.colorbar(im, ax=ax, label="Feature value")
plt.tight_layout()
plt.show()

print("Each row is one atom's 40-dim one-hot feature vector.")
print("Bright cells show which category is 'on' for each feature group.")

In [ ]:
# Visualization 2: Adjacency matrix colored by bond type
# Build a dense adjacency matrix where the value encodes bond type
BOND_LABELS = ["SINGLE", "DOUBLE", "TRIPLE", "AROMATIC", "self-loop"]
adj = np.zeros((N, N), dtype=np.float32)

for e in range(E):
    src, dst = edge_index[0, e], edge_index[1, e]
    bond_type_idx = np.argmax(edge_attr[e])  # 0=SINGLE, 1=DOUBLE, ..., 4=self-loop
    adj[src, dst] = bond_type_idx + 1  # +1 so that 0 = no edge

fig, ax = plt.subplots(figsize=(7, 6))
colors = ["white", "#4CAF50", "#FF9800", "#F44336", "#9C27B0", "#607D8B"]
cmap = mcolors.ListedColormap(colors)
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

im = ax.imshow(adj, cmap=cmap, norm=norm, interpolation="nearest")
ax.set_xticks(range(N))
ax.set_yticks(range(N))
ax.set_xticklabels([f"{i}:{s}" for i, s in enumerate(atom_symbols)], rotation=45, ha="right", fontsize=8)
ax.set_yticklabels([f"{i}:{s}" for i, s in enumerate(atom_symbols)], fontsize=8)
ax.set_xlabel("Destination atom")
ax.set_ylabel("Source atom")
ax.set_title(f"Adjacency matrix  [{N}, {N}]  (colored by bond type)")

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4, 5])
cbar.ax.set_yticklabels(["none"] + BOND_LABELS)
plt.tight_layout()
plt.show()

print(f"The graph has {E} directed edges:")
print(f"  {2*num_bonds} bond edges (each bond stored bidirectionally)")
print(f"  {N} self-loops (every atom connects to itself)")

---
## Section 3: Input Projection

The raw atom features are **sparse one-hot vectors** (40 dimensions, mostly zeros). The input
projection is a simple linear layer that maps them into a **dense 256-dimensional embedding
space** where the model can learn useful representations:

$$\mathbf{x}_i' = \mathbf{W} \cdot \mathbf{x}_i + \mathbf{b}$$

where $\mathbf{W} \in \mathbb{R}^{256 \times 40}$ and $\mathbf{b} \in \mathbb{R}^{256}$.

There is **no activation function** here — it is a raw linear transform. The GAT layers
that follow will add nonlinearity.

In [ ]:
# Convert to tensors
x = torch.from_numpy(node_feat)          # shape: [13, 40]
ei = torch.from_numpy(edge_index)        # shape: [2, E]
ea = torch.from_numpy(edge_attr)         # shape: [E, 5]

with torch.no_grad():
    W_proj = model.input_proj.weight      # shape: [256, 40]
    b_proj = model.input_proj.bias        # shape: [256]
    print(f"Input projection weights:")
    print(f"  W shape: {list(W_proj.shape)}")
    print(f"  b shape: {list(b_proj.shape)}")
    print()

    # Manual computation: x_proj = x @ W^T + b
    x_proj_manual = x @ W_proj.T + b_proj  # shape: [13, 256]

    # Using the model's layer
    x_proj_model = model.input_proj(x)      # shape: [13, 256]

    # Verify they match
    diff = (x_proj_manual - x_proj_model).abs().max().item()
    print(f"Shape transition: [{N}, 40] → [{N}, 256]")
    print(f"Manual vs model max diff: {diff:.2e} (should be ~0)")

# Visualization 3: Sparse input vs dense projected embedding
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 3]})

ax1.imshow(node_feat, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax1.set_title(f"Input features  [{N}, 40]\n(sparse one-hot)")
ax1.set_ylabel("Atom")
ax1.set_xlabel("Feature dim")
ax1.set_yticks(range(N))
ax1.set_yticklabels(atom_symbols)

im2 = ax2.imshow(x_proj_manual.numpy(), aspect="auto", cmap="RdBu_r", interpolation="nearest")
ax2.set_title(f"Projected embeddings  [{N}, 256]\n(dense, learned)")
ax2.set_ylabel("Atom")
ax2.set_xlabel("Embedding dim")
ax2.set_yticks(range(N))
ax2.set_yticklabels(atom_symbols)
plt.colorbar(im2, ax=ax2, label="Value")

plt.tight_layout()
plt.show()

print("Left: sparse input (most values are 0).")
print("Right: dense projection (every dimension carries information).")
print("The linear projection 'unpacks' the one-hot categories into a rich embedding space.")

---
## Section 4: Graph Attention Layer Deep Dive

This is the core of the model. Each GAT layer updates every node's embedding by **attending
to its neighbors**. The key idea: not all neighbors are equally important. The attention
mechanism learns **which neighbors matter most** for each node.

The computation has 4 steps:

### Step 1: Linear transform (per head)
Each of the 4 heads gets its own view of the node features via a shared linear projection:
$$\mathbf{h}'_i = \mathbf{W} \cdot \mathbf{h}_i \quad \in \mathbb{R}^{1024}$$
where $\mathbf{W} \in \mathbb{R}^{1024 \times 256}$. The output is then reshaped to $[N, 4, 256]$
so each head sees a 256-dim slice.

### Step 2: Attention scores
For each edge $(i \to j)$:
$$e_{ij} = \text{LeakyReLU}\left(\mathbf{a}_{\text{src}} \cdot \mathbf{h}'_i + \mathbf{a}_{\text{dst}} \cdot \mathbf{h}'_j + \mathbf{a}_{\text{edge}} \cdot \mathbf{e}_{ij}\right)$$
where $\mathbf{a}_{\text{src}}, \mathbf{a}_{\text{dst}} \in \mathbb{R}^{256}$ per head and $\mathbf{a}_{\text{edge}} \in \mathbb{R}^{5}$ (one scalar per head from a linear projection of the 5-dim edge feature).

### Step 3: Edge softmax (attention weights)
Normalize scores over all incoming edges for each destination node $j$:
$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(j)} \exp(e_{kj})}$$
This is a softmax **grouped by destination node** — each node's incoming attention weights sum to 1.

### Step 4: Weighted aggregation + multi-head mean
$$\mathbf{h}_j^{(\text{head}_k)} = \sum_{i \in \mathcal{N}(j)} \alpha_{ij} \cdot \mathbf{h}'_i \quad \in \mathbb{R}^{256}$$
$$\mathbf{h}_j = \frac{1}{4} \sum_{k=1}^{4} \mathbf{h}_j^{(\text{head}_k)} + \mathbf{b} \quad \in \mathbb{R}^{256}$$
Each head produces a full 256-dim output, and they are **averaged** (not concatenated).
This is different from the original GAT paper which concatenates heads — here averaging
keeps the output dimension constant at 256.

In [ ]:
# Extract GAT layer 0 parameters
gat0 = model.gat_layers[0]
with torch.no_grad():
    W_lin = gat0.lin.weight       # shape: [heads*out_dim, in_dim] = [1024, 256]
    a_src = gat0.att_src          # shape: [heads, out_dim] = [4, 256]
    a_dst = gat0.att_dst          # shape: [heads, out_dim] = [4, 256]
    bias  = gat0.bias             # shape: [out_dim] = [256]
    W_edge = gat0.edge_proj.weight  # shape: [heads, edge_dim] = [4, 5]

print("GAT layer 0 parameters:")
print(f"  W_lin     (node linear):   {list(W_lin.shape)}  — projects each node: 256 → 4×256")
print(f"  att_src   (source attn):   {list(a_src.shape)}  — one 256-dim vector per head")
print(f"  att_dst   (dest attn):     {list(a_dst.shape)}  — one 256-dim vector per head")
print(f"  edge_proj (edge linear):   {list(W_edge.shape)} — projects 5-dim edge feat → 4 scalars")
print(f"  bias      (output bias):   {list(bias.shape)}")
print(f"\n  Total params in this layer: {sum(p.numel() for p in gat0.parameters()):,}")

In [ ]:
# Step-by-step manual computation through GAT layer 0
with torch.no_grad():
    # Start from the projected node features
    h_in = x_proj_manual.clone()  # shape: [13, 256]
    print("=== Step 1: Linear transform ===")

    h = h_in @ W_lin.T                                  # shape: [13, 1024]
    h = h.view(N, gat0.heads, gat0.out_dim)             # shape: [13, 4, 256]
    print(f"  h_in:  {list(h_in.shape)}")
    print(f"  W_lin: {list(W_lin.shape)}")
    print(f"  h = (h_in @ W^T).view(N, heads, out_dim): {list(h.shape)}")

    print("\n=== Step 2: Attention scores ===")
    # Compute source and destination attention terms
    att_src_scores = (h * a_src).sum(dim=-1)            # shape: [13, 4]
    att_dst_scores = (h * a_dst).sum(dim=-1)            # shape: [13, 4]
    print(f"  att_src per node: {list(att_src_scores.shape)}  (dot product of h with a_src for each head)")
    print(f"  att_dst per node: {list(att_dst_scores.shape)}")

    # Gather along edges
    src, dst = ei
    edge_logits = (
        att_src_scores[src]       # source node's score, shape: [E, 4]
        + att_dst_scores[dst]     # dest node's score,   shape: [E, 4]
        + (ea @ W_edge.T)         # edge feature contrib, shape: [E, 4]
    )                             # total: [E, 4]
    print(f"  raw edge logits: {list(edge_logits.shape)}")

    edge_logits = F.leaky_relu(edge_logits, 0.2)        # shape: [E, 4]
    print(f"  after LeakyReLU(0.2): {list(edge_logits.shape)}")
    print(f"  logit range: [{edge_logits.min():.3f}, {edge_logits.max():.3f}]")

In [ ]:
# Step 3: Edge softmax (manual implementation with numerical stability)
with torch.no_grad():
    print("=== Step 3: Edge softmax ===")
    print("For each destination node j, normalize attention over all incoming edges:")
    print("  α_ij = exp(e_ij) / Σ_{k∈N(j)} exp(e_kj)")
    print()

    # Manual edge softmax (matching the model's implementation)
    alpha_manual = edge_softmax(dst, edge_logits, N)    # shape: [E, 4]

    # Verify: attention weights for each destination node should sum to 1
    for node_j in range(min(3, N)):  # Show first 3 nodes
        incoming_mask = (dst == node_j)
        incoming_alphas = alpha_manual[incoming_mask]     # shape: [num_incoming, 4]
        sums = incoming_alphas.sum(dim=0)
        src_atoms = src[incoming_mask].tolist()
        print(f"  Node {node_j} ({atom_symbols[node_j]}): "
              f"{incoming_mask.sum().item()} incoming edges from atoms {src_atoms}")
        print(f"    Attention sums per head: {sums.tolist()} (should all be ~1.0)")

    print(f"\n  Attention weights shape: {list(alpha_manual.shape)}")
    print(f"  Range: [{alpha_manual.min():.4f}, {alpha_manual.max():.4f}]")

In [ ]:
# Step 4: Weighted aggregation
with torch.no_grad():
    print("=== Step 4: Weighted aggregation ===")
    print("  h_j = Σ_{i∈N(j)} α_ij · h'_i  (per head)")
    print("  Then average over heads and add bias.")
    print()

    # Weighted messages: each edge carries source node's transformed features × attention weight
    messages = h[src] * alpha_manual.unsqueeze(-1)       # shape: [E, 4, 256]
    print(f"  messages (h[src] * α): {list(messages.shape)}")

    # Aggregate by destination node
    agg = torch.zeros(N, gat0.heads, gat0.out_dim)
    agg.index_add_(0, dst, messages)                    # shape: [13, 4, 256]
    print(f"  aggregated per node:   {list(agg.shape)}")

    # Average over heads + bias
    gat_out_manual = agg.mean(dim=1) + bias             # shape: [13, 256]
    print(f"  after head mean + bias: {list(gat_out_manual.shape)}")

    # Compare with model's forward pass
    gat_out_model = gat0(h_in, ei, ea)                  # shape: [13, 256]
    diff = (gat_out_manual - gat_out_model).abs().max().item()
    print(f"\n  Manual vs model max diff: {diff:.2e} (should be ~0)")

In [ ]:
# Visualization 4: Attention weight heatmaps (one per head)
with torch.no_grad():
    # Build dense attention matrices [N, N] per head
    attn_matrices = np.zeros((gat0.heads, N, N))
    for e_idx in range(E):
        s, d = src[e_idx].item(), dst[e_idx].item()
        for head in range(gat0.heads):
            attn_matrices[head, s, d] = alpha_manual[e_idx, head].item()

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for head in range(4):
        ax = axes[head]
        im = ax.imshow(attn_matrices[head], cmap="Blues", vmin=0, interpolation="nearest")
        ax.set_title(f"Head {head}")
        ax.set_xticks(range(N))
        ax.set_yticks(range(N))
        ax.set_xticklabels(atom_symbols, fontsize=7, rotation=45)
        ax.set_yticklabels(atom_symbols, fontsize=7)
        if head == 0:
            ax.set_ylabel("Source atom")
        ax.set_xlabel("Destination atom")
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.suptitle("GAT Layer 0: Attention weights α  (source → destination)\n"
                 "Each column sums to 1 (softmax over incoming edges per dest node)",
                 fontsize=12)
    plt.tight_layout()
    plt.show()

    print("Each head learns a different attention pattern.")
    print("Bright cells = high attention. The diagonal shows self-loop attention.")
    print("Off-diagonal entries show how much each atom 'listens to' its bonded neighbors.")

---
## Section 5: Full GAT Block — Residual + Norm + Activation

The raw GAT output is just one piece of a full block. The complete computation is:

```
residual = x
x = GAT(x, edge_index, edge_attr)   # attention-based message passing
x = LayerNorm(x)                     # normalize
x = ELU(x)                           # nonlinear activation
x = Dropout(x)                       # regularization (disabled at eval time)
x = x + residual                     # skip connection
```

**Why residual connections?** Without them, GNN layers tend to make all node embeddings
converge to the same vector ("over-smoothing"). The skip connection lets the model preserve
the original node identity while adding neighbor information.

**Why LayerNorm?** Normalizes each node's 256-dim vector to have mean 0 and variance 1,
then applies learned scale ($\gamma$) and shift ($\beta$). This stabilizes training by
preventing the embeddings from exploding or vanishing across layers.

**Why ELU over ReLU?** ELU($x$) = $x$ if $x > 0$, else $\alpha(e^x - 1)$. Unlike ReLU,
ELU has nonzero gradients for negative inputs, which helps avoid "dead neurons".

In [ ]:
with torch.no_grad():
    # We already have gat_out_manual from Section 4
    norm0 = model.layer_norms[0]
    gamma = norm0.weight  # shape: [256]
    beta = norm0.bias     # shape: [256]

    print("=== LayerNorm ===")
    print(f"  gamma (scale): {list(gamma.shape)}, beta (shift): {list(beta.shape)}")

    # Manual LayerNorm: y = γ · (x - μ) / √(σ² + ε) + β
    eps = 1e-5
    mean = gat_out_manual.mean(dim=-1, keepdim=True)     # shape: [13, 1]
    var = gat_out_manual.var(dim=-1, keepdim=True, unbiased=False)  # shape: [13, 1]
    normed_manual = gamma * (gat_out_manual - mean) / torch.sqrt(var + eps) + beta  # shape: [13, 256]

    # Using PyTorch's LayerNorm
    normed_model = norm0(gat_out_manual)                 # shape: [13, 256]

    diff = (normed_manual - normed_model).abs().max().item()
    print(f"  Manual vs model max diff: {diff:.2e}")
    print(f"  Output shape: {list(normed_model.shape)}")

    # Apply ELU
    print("\n=== ELU activation ===")
    print("  ELU(x) = x if x > 0, else α·(exp(x) - 1)  where α=1.0")
    activated = F.elu(normed_model)                      # shape: [13, 256]
    pct_positive = (activated > 0).float().mean().item() * 100
    print(f"  {pct_positive:.1f}% of values are positive (passed through unchanged)")

    # No dropout at eval time (model.eval() disables it)

    # Residual connection
    print("\n=== Residual connection ===")
    block_output = activated + x_proj_manual              # shape: [13, 256]
    print(f"  output = ELU(LayerNorm(GAT(x))) + x")
    print(f"  Input shape:  {list(x_proj_manual.shape)}")
    print(f"  Output shape: {list(block_output.shape)}")
    print(f"  (Same shape — residual connections require matching dimensions)")

In [ ]:
# Verify full block matches model's computation
with torch.no_grad():
    # Run the full first block through the model
    x_check = x_proj_manual.clone()
    residual = x_check
    x_check = model.gat_layers[0](x_check, ei, ea)
    x_check = model.layer_norms[0](x_check)
    x_check = F.elu(x_check)
    # No dropout at eval
    x_check = x_check + residual

    diff = (block_output - x_check).abs().max().item()
    print(f"Full block manual vs model max diff: {diff:.2e} (should be ~0)")
    print(f"Block output shape: {list(block_output.shape)}")

---
## Section 6: GAT Layers 1 & 2

Layers 1 and 2 are structurally identical to layer 0: same hidden dimension (256),
same number of heads (4), same 256-dim per head (averaged). Each successive layer lets every node
incorporate information from **further-away** neighbors:

- After layer 0: each node knows about its **1-hop** neighborhood
- After layer 1: each node knows about its **2-hop** neighborhood
- After layer 2: each node knows about its **3-hop** neighborhood

We expect the cosine similarity between node embeddings to increase with more layers
(nodes become more similar as they share more global information).

In [ ]:
# Run all 3 layers, collecting embeddings at each stage
with torch.no_grad():
    layer_embeddings = [x_proj_manual.clone()]  # Layer 0 input
    x_running = x_proj_manual.clone()

    for i, (gat, norm) in enumerate(zip(model.gat_layers, model.layer_norms)):
        residual = x_running
        x_running = gat(x_running, ei, ea)
        x_running = norm(x_running)
        x_running = F.elu(x_running)
        x_running = x_running + residual
        layer_embeddings.append(x_running.clone())
        print(f"After GAT block {i}: {list(x_running.shape)}")

    print(f"\nCollected {len(layer_embeddings)} embedding snapshots (input + 3 layer outputs)")

# Visualization 5: Cosine similarity matrices across layers
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
titles = ["Input projection\n(before any GAT)", "After GAT block 0",
          "After GAT block 1", "After GAT block 2"]

for idx, (emb, title) in enumerate(zip(layer_embeddings, titles)):
    # Cosine similarity: normalize rows, then compute dot product
    emb_norm = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    cos_sim = (emb_norm @ emb_norm.T).numpy()  # shape: [13, 13]

    ax = axes[idx]
    im = ax.imshow(cos_sim, cmap="RdYlBu_r", vmin=-1, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.set_xticks(range(N))
    ax.set_yticks(range(N))
    ax.set_xticklabels(atom_symbols, fontsize=7, rotation=45)
    ax.set_yticklabels(atom_symbols, fontsize=7)
    if idx == 0:
        ax.set_ylabel("Atom")

plt.colorbar(im, ax=axes[-1], fraction=0.046, label="Cosine similarity")
plt.suptitle("Node embedding similarity evolving across GAT layers\n"
             "(darker red = more similar embeddings)", fontsize=12)
plt.tight_layout()
plt.show()

# Summary statistics
for idx, (emb, title) in enumerate(zip(layer_embeddings, titles)):
    emb_norm = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    cos_sim = (emb_norm @ emb_norm.T)
    # Off-diagonal mean
    mask = ~torch.eye(N, dtype=torch.bool)
    mean_sim = cos_sim[mask].mean().item()
    print(f"{title.split(chr(10))[0]:30s}  mean off-diagonal cosine sim: {mean_sim:.4f}")

---
## Section 7: Mean Pooling — Nodes → Graph

After 3 GAT layers, we have a 256-dim embedding for each of the 13 atoms. But we need a
**single fixed-size vector** for the entire molecule (to feed to the classifier).

**Global mean pooling** is the simplest approach: average all node embeddings:
$$\mathbf{h}_{\text{graph}} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{h}_i$$

For aspirin: $[13, 256] \to [1, 256]$.

In [ ]:
with torch.no_grad():
    final_node_embs = layer_embeddings[-1]  # shape: [13, 256]

    # Manual mean pooling
    graph_embed_manual = final_node_embs.mean(dim=0, keepdim=True)  # shape: [1, 256]
    print(f"Node embeddings: {list(final_node_embs.shape)}")
    print(f"Graph embedding: {list(graph_embed_manual.shape)}")
    print(f"  Mean value: {graph_embed_manual.mean().item():.4f}")
    print(f"  Std:        {graph_embed_manual.std().item():.4f}")
    print(f"  Min:        {graph_embed_manual.min().item():.4f}")
    print(f"  Max:        {graph_embed_manual.max().item():.4f}")

# Visualization 6: Graph embedding bar chart
embed_np = graph_embed_manual.squeeze(0).numpy()  # shape: [256]

fig, ax = plt.subplots(figsize=(14, 4))
colors = ["#e74c3c" if v > 0 else "#3498db" for v in embed_np]
ax.bar(range(256), embed_np, color=colors, width=1.0, edgecolor="none")
ax.set_xlabel("Embedding dimension")
ax.set_ylabel("Value")
ax.set_title(f"Graph embedding vector  [1, 256]\n"
             f"(mean={embed_np.mean():.3f}, std={embed_np.std():.3f})")
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_xlim(-1, 256)

# Highlight top-5 most activated dimensions
top_k = 5
top_indices = np.argsort(np.abs(embed_np))[-top_k:]
for idx in top_indices:
    ax.annotate(f"dim {idx}", xy=(idx, embed_np[idx]),
                fontsize=7, ha="center",
                xytext=(0, 10 if embed_np[idx] > 0 else -15),
                textcoords="offset points")

plt.tight_layout()
plt.show()

print(f"Red bars = positive, blue bars = negative.")
print(f"Top-{top_k} most activated dimensions annotated.")
print(f"This 256-dim vector is the molecule's 'fingerprint' — it goes to the classifier next.")

---
## Section 8: Classification Head

The classification head is a small MLP (multi-layer perceptron) that maps the 256-dim
graph embedding to a single logit:

$$\text{logit} = \mathbf{W}_2 \cdot \text{ReLU}(\mathbf{W}_1 \cdot \mathbf{h}_{\text{graph}} + \mathbf{b}_1) + \mathbf{b}_2$$

Where:
- $\mathbf{W}_1 \in \mathbb{R}^{128 \times 256}$, $\mathbf{b}_1 \in \mathbb{R}^{128}$
- $\mathbf{W}_2 \in \mathbb{R}^{1 \times 128}$, $\mathbf{b}_2 \in \mathbb{R}^{1}$

The logit is then passed through **sigmoid** to get a probability:
$$P(\text{PKS}) = \sigma(\text{logit}) = \frac{1}{1 + e^{-\text{logit}}}$$

In [ ]:
with torch.no_grad():
    # Extract classification head parameters
    W1 = model.cls_head[0].weight  # shape: [128, 256]
    b1 = model.cls_head[0].bias    # shape: [128]
    W2 = model.cls_head[2].weight  # shape: [1, 128]
    b2 = model.cls_head[2].bias    # shape: [1]

    print("Classification head parameters:")
    print(f"  W1: {list(W1.shape)}, b1: {list(b1.shape)}")
    print(f"  W2: {list(W2.shape)}, b2: {list(b2.shape)}")
    print(f"  Total params: {W1.numel() + b1.numel() + W2.numel() + b2.numel():,}")
    print()

    # Step through the MLP
    h1 = graph_embed_manual @ W1.T + b1       # shape: [1, 128]
    print(f"Step 1 — Linear(256→128): {list(h1.shape)}")

    h1_relu = F.relu(h1)                       # shape: [1, 128]
    pct_active = (h1_relu > 0).float().mean().item() * 100
    print(f"Step 2 — ReLU:            {list(h1_relu.shape)}  ({pct_active:.0f}% neurons active)")

    logit_manual = h1_relu @ W2.T + b2         # shape: [1, 1]
    print(f"Step 3 — Linear(128→1):   {list(logit_manual.shape)}")
    print(f"  Logit value: {logit_manual.item():.4f}")

    prob_manual = torch.sigmoid(logit_manual)   # shape: [1, 1]
    print(f"  Sigmoid(logit): {prob_manual.item():.4f}")
    label = "PKS" if prob_manual.item() >= 0.5 else "non-PKS"
    print(f"  Prediction: {label}")

In [ ]:
# Visualization 7: Sigmoid curve with the model's logit marked
fig, ax = plt.subplots(figsize=(8, 5))

z = np.linspace(-8, 8, 500)
sigmoid = 1 / (1 + np.exp(-z))
ax.plot(z, sigmoid, "b-", linewidth=2, label="σ(z) = 1 / (1 + e⁻ᶻ)")
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Decision boundary (0.5)")
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.3)

# Mark aspirin's logit
logit_val = logit_manual.item()
prob_val = prob_manual.item()
ax.plot(logit_val, prob_val, "ro", markersize=12, zorder=5,
        label=f"Aspirin: logit={logit_val:.2f}, P(PKS)={prob_val:.4f}")
ax.annotate(f"  Aspirin\n  logit={logit_val:.2f}",
            xy=(logit_val, prob_val), fontsize=10,
            xytext=(logit_val + 1.5, prob_val + 0.1),
            arrowprops=dict(arrowstyle="->", color="red"))

ax.set_xlabel("Logit (z)", fontsize=12)
ax.set_ylabel("P(PKS) = σ(z)", fontsize=12)
ax.set_title("Sigmoid activation: logit → probability")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

print("The model is trained with BCEWithLogitsLoss (pos_weight=2.0).")
print("pos_weight=2.0 penalizes false negatives 2× more than false positives,")
print("compensating for the ~33% PKS / ~67% non-PKS class imbalance in training data.")

---
## Section 9: End-to-End Verification

We expect our manual forward pass to **exactly match** the model's `forward()` method.
We also compare predictions on aspirin (expected: non-PKS) vs a known PKS product
from the training data (expected: PKS).

In [ ]:
# --- Aspirin: compare manual vs model ---
with torch.no_grad():
    batch_t = torch.zeros(N, dtype=torch.long)  # single graph → all nodes in batch 0
    logit_model, embed_model = model(x, ei, batch_t, ea)

    print("=== Aspirin (expected: non-PKS) ===")
    print(f"  Manual logit:  {logit_manual.item():.6f}")
    print(f"  Model logit:   {logit_model.item():.6f}")
    print(f"  Max diff:      {abs(logit_manual.item() - logit_model.item()):.2e}")
    print(f"  Manual P(PKS): {prob_manual.item():.6f}")
    print(f"  Model P(PKS):  {torch.sigmoid(logit_model).item():.6f}")
    print()

# --- PKS product from training data ---
PKS_SMILES = "CCC(=O)CC(=O)O"  # a small PKS product
nf_pks, ei_pks, ea_pks = smiles_to_graph(PKS_SMILES)
x_pks = torch.from_numpy(nf_pks)
ei_pks_t = torch.from_numpy(ei_pks)
ea_pks_t = torch.from_numpy(ea_pks)
batch_pks = torch.zeros(nf_pks.shape[0], dtype=torch.long)

with torch.no_grad():
    logit_pks, embed_pks = model(x_pks, ei_pks_t, batch_pks, ea_pks_t)
    prob_pks = torch.sigmoid(logit_pks).item()

    print(f"=== PKS product: {PKS_SMILES} (expected: PKS) ===")
    print(f"  Logit:  {logit_pks.item():.6f}")
    print(f"  P(PKS): {prob_pks:.6f}")
    print()

    print("=== Side-by-side comparison ===")
    print(f"  {'Molecule':<30} {'Logit':>10} {'P(PKS)':>10} {'Prediction':>12}")
    print(f"  {'-'*62}")
    asp_prob = torch.sigmoid(logit_model).item()
    print(f"  {'Aspirin (non-PKS)':<30} {logit_model.item():>10.4f} {asp_prob:>10.4f} {'PKS' if asp_prob >= 0.5 else 'non-PKS':>12}")
    print(f"  {PKS_SMILES + ' (PKS)':<30} {logit_pks.item():>10.4f} {prob_pks:>10.4f} {'PKS' if prob_pks >= 0.5 else 'non-PKS':>12}")

---
## Section 10: Understanding the Loss

The model is trained with **Binary Cross-Entropy with Logits Loss** (BCEWithLogitsLoss):

$$\mathcal{L} = -\left[ y \cdot \log(\sigma(z)) \cdot w_{\text{pos}} + (1 - y) \cdot \log(1 - \sigma(z)) \right]$$

Where:
- $z$ = raw logit from the model
- $y$ = true label (1 for PKS, 0 for non-PKS)
- $\sigma(z)$ = sigmoid(logit) = predicted probability
- $w_{\text{pos}} = 2.0$ = positive class weight

**Why pos_weight=2.0?** PKS products are ~33% of the training data (1 PKS + 2 non-PKS per
triplet). Without class weighting, the model could get 67% accuracy by always predicting
non-PKS. `pos_weight=2.0` makes the loss 2× larger when the model misclassifies a true PKS
molecule, balancing the incentive.

In [ ]:
with torch.no_grad():
    pos_weight = torch.tensor([2.0])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # --- Aspirin (label=0, non-PKS) ---
    logit_asp = logit_model.clone()  # shape: [1, 1]
    label_asp = torch.tensor([[0.0]])

    # Manual loss: L = -[(1-y)·log(1-σ(z)) + y·log(σ(z))·w]
    sigma_asp = torch.sigmoid(logit_asp)
    loss_asp_manual = -(label_asp * torch.log(sigma_asp + 1e-8) * pos_weight
                        + (1 - label_asp) * torch.log(1 - sigma_asp + 1e-8))
    loss_asp_pytorch = criterion(logit_asp, label_asp)

    print("=== Aspirin (label=0, non-PKS) ===")
    print(f"  Logit: {logit_asp.item():.4f}, σ(logit): {sigma_asp.item():.4f}")
    print(f"  Manual loss:  {loss_asp_manual.item():.6f}")
    print(f"  PyTorch loss: {loss_asp_pytorch.item():.6f}")
    print(f"  Diff:         {abs(loss_asp_manual.item() - loss_asp_pytorch.item()):.2e}")
    print()

    # --- PKS product (label=1, PKS) ---
    logit_pks_val = logit_pks.clone()  # shape: [1, 1]
    label_pks_val = torch.tensor([[1.0]])

    sigma_pks = torch.sigmoid(logit_pks_val)
    loss_pks_manual = -(label_pks_val * torch.log(sigma_pks + 1e-8) * pos_weight
                        + (1 - label_pks_val) * torch.log(1 - sigma_pks + 1e-8))
    loss_pks_pytorch = criterion(logit_pks_val, label_pks_val)

    print(f"=== PKS product (label=1, PKS) ===")
    print(f"  Logit: {logit_pks_val.item():.4f}, σ(logit): {sigma_pks.item():.4f}")
    print(f"  Manual loss:  {loss_pks_manual.item():.6f}")
    print(f"  PyTorch loss: {loss_pks_pytorch.item():.6f}")
    print(f"  Diff:         {abs(loss_pks_manual.item() - loss_pks_pytorch.item()):.2e}")
    print()

    print("Note: small diffs are expected due to BCEWithLogitsLoss using a numerically")
    print("stable formula: max(z,0) - z*y + log(1+exp(-|z|)) rather than log(sigmoid).")

---
## Section 11: Batched Processing

In practice, we process many molecules at once for efficiency. PyG-style batching works by
**merging multiple graphs into one disconnected super-graph**:

1. Concatenate all node features: `[N1+N2+..., 40]`
2. Concatenate edge indices, but **offset** each graph's indices by the cumulative node count
3. Create a `batch` vector that maps each node to its originating graph

The model then processes this as a single graph. Mean pooling uses the `batch` vector to
average nodes within each molecule separately.

In [ ]:
# collate_graphs function (from scripts/08_train_gnn_classifier.py)
@dataclass
class GraphSample:
    node_feat: np.ndarray
    edge_index: np.ndarray
    edge_attr: np.ndarray
    label: int

def collate_graphs(batch):
    node_feats, edge_indices, edge_attrs, batch_index, labels = [], [], [], [], []
    offset = 0
    for graph in batch:
        x_g = torch.from_numpy(graph.node_feat)
        ei_g = torch.from_numpy(graph.edge_index) + offset
        ea_g = torch.from_numpy(graph.edge_attr)
        n = x_g.size(0)
        node_feats.append(x_g)
        edge_indices.append(ei_g)
        edge_attrs.append(ea_g)
        batch_index.append(torch.full((n,), len(labels), dtype=torch.long))
        labels.append(graph.label)
        offset += n
    return {
        "node_feat": torch.cat(node_feats, dim=0),
        "edge_index": torch.cat(edge_indices, dim=1),
        "edge_attr": torch.cat(edge_attrs, dim=0),
        "batch": torch.cat(batch_index, dim=0),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


# Create two graph samples
nf_asp, ei_asp, ea_asp = smiles_to_graph(ASPIRIN_SMILES)
nf_pks, ei_pks, ea_pks = smiles_to_graph(PKS_SMILES)

sample_asp = GraphSample(nf_asp, ei_asp, ea_asp, label=0)
sample_pks = GraphSample(nf_pks, ei_pks, ea_pks, label=1)

batched = collate_graphs([sample_asp, sample_pks])

N_asp = nf_asp.shape[0]
N_pks = nf_pks.shape[0]
print(f"Aspirin:     {N_asp} atoms, {ei_asp.shape[1]} edges")
print(f"PKS product: {N_pks} atoms, {ei_pks.shape[1]} edges")
print()
print(f"Batched tensors:")
print(f"  node_feat:  {list(batched['node_feat'].shape)}  (= {N_asp} + {N_pks} nodes)")
print(f"  edge_index: {list(batched['edge_index'].shape)}  (= {ei_asp.shape[1]} + {ei_pks.shape[1]} edges)")
print(f"  edge_attr:  {list(batched['edge_attr'].shape)}")
print(f"  batch:      {list(batched['batch'].shape)}  values: {batched['batch'].tolist()}")
print(f"  labels:     {batched['labels'].tolist()}")
print()

# Show edge index offset
print(f"Edge index ranges:")
print(f"  Aspirin edges: node indices 0–{N_asp-1}")
print(f"  PKS edges:     node indices {N_asp}–{N_asp+N_pks-1} (offset by {N_asp})")
print()

# Run batched inference
with torch.no_grad():
    logits_batched, embeds_batched = model(
        batched["node_feat"], batched["edge_index"],
        batched["batch"], batched["edge_attr"]
    )
    probs_batched = torch.sigmoid(logits_batched)

    print(f"Batched output:")
    print(f"  Logits shape:     {list(logits_batched.shape)}  (one per molecule)")
    print(f"  Embeddings shape: {list(embeds_batched.shape)}")
    print()
    print(f"  Aspirin:     logit={logits_batched[0].item():.4f}, P(PKS)={probs_batched[0].item():.4f}")
    print(f"  PKS product: logit={logits_batched[1].item():.4f}, P(PKS)={probs_batched[1].item():.4f}")

---
## Section 12: Summary

### Layer-by-layer shape recap

| Stage | Operation | Shape | Notes |
|-------|-----------|-------|-------|
| Input | SMILES → graph | `x=[N,40]`, `ei=[2,E]`, `ea=[E,5]` | RDKit featurization |
| Projection | `Linear(40→256)` | `[N, 40] → [N, 256]` | No activation |
| GAT block 0 | Attention + residual + LN + ELU | `[N, 256] → [N, 256]` | 4 heads × 256 dim, averaged |
| GAT block 1 | Attention + residual + LN + ELU | `[N, 256] → [N, 256]` | 4 heads × 256 dim, averaged |
| GAT block 2 | Attention + residual + LN + ELU | `[N, 256] → [N, 256]` | 4 heads × 256 dim, averaged |
| Pooling | Global mean | `[N, 256] → [1, 256]` | Average all nodes |
| Classifier | `Linear(256→128) → ReLU → Linear(128→1)` | `[1, 256] → [1, 1]` | Raw logit |
| Output | Sigmoid | `[1, 1] → [1, 1]` | P(PKS) ∈ [0, 1] |

### Key takeaways

1. **Graph representation is natural for molecules** — atoms are nodes, bonds are edges,
   and the GNN processes this structure directly (no fixed-size fingerprint needed).

2. **Attention learns which neighbors matter** — each GAT head develops a different
   attention pattern, allowing the model to focus on chemically relevant substructures.

3. **Residual connections are essential** — without them, 3 layers of message passing
   would over-smooth all node embeddings into a single indistinguishable vector.

4. **Mean pooling is simple but effective** — averaging all node embeddings gives a
   fixed-size molecular fingerprint that captures global properties.

5. **The full pipeline is differentiable** — gradients flow from the BCE loss through
   the classifier, pooling, all GAT layers, and into the input projection, so every
   component is jointly optimized end-to-end.